# 03 — Evaluación de Métricas

Evaluación detallada del modelo entrenado:
- mAP, precision, recall por clase
- Matriz de confusión
- Comparativa entre modelos
- Análisis de errores

In [ ]:
# === Setup para Google Colab ===
import os

if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/neurodrive
    !pip install -q ultralytics
    print('✅ Entorno Colab configurado')
else:
    print('✅ Entorno local detectado')

In [ ]:
# === Configuración ===

# Modelos a evaluar (agregar rutas de modelos entrenados)
MODELS = [
    'results/baseline_v1/weights/best.pt',
    # 'results/improved_v1/weights/best.pt',  # Descomentar cuando exista
]

DATASET_YAML = 'configs/datasets/tier1_road_vehicles.yaml'
OUTPUT_DIR = 'results/evaluacion'

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Evaluación Individual

Evaluamos cada modelo contra el dataset de validación.

In [ ]:
from src.models.evaluate import evaluate_model

all_metrics = []
for model_path in MODELS:
    if Path(model_path).exists():
        metrics = evaluate_model(model_path, DATASET_YAML)
        all_metrics.append(metrics)
    else:
        print(f'⚠️ Modelo no encontrado: {model_path}')

if all_metrics:
    df = pd.DataFrame(all_metrics)
    print('\n📊 Tabla Comparativa:')
    display(df)

## Matriz de Confusión

La matriz de confusión muestra cómo el modelo clasifica cada tipo de vehículo.

In [ ]:
# Mostrar matriz de confusión del mejor modelo
from IPython.display import Image, display

for model_path in MODELS:
    run_dir = Path(model_path).parent.parent
    cm_path = run_dir / 'confusion_matrix.png'
    if cm_path.exists():
        print(f'Modelo: {Path(model_path).parent.parent.name}')
        display(Image(filename=str(cm_path), width=600))

## Comparativa Visual

Si hay múltiples modelos, comparamos sus métricas visualmente.

In [ ]:
if len(all_metrics) > 1:
    from src.models.evaluate import plot_metrics_comparison
    
    output_path = f'{OUTPUT_DIR}/comparativa_metricas.png'
    plot_metrics_comparison(df, output_path)
    display(Image(filename=output_path, width=800))
else:
    print('ℹ️ Se necesitan al menos 2 modelos para comparar')
    if all_metrics:
        # Gráfica de un solo modelo
        m = all_metrics[0]
        metrics_names = ['mAP50', 'mAP50_95', 'precision', 'recall']
        values = [m[k] for k in metrics_names]
        
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(metrics_names, values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
        ax.set_ylim(0, 1)
        ax.set_title(f"Métricas — {m['modelo']}")
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center')
        plt.tight_layout()
        plt.show()

## Análisis de Errores

Revisamos ejemplos donde el modelo falla (falsos positivos, falsos negativos, confusiones entre clases).

> **Output esperado**: Imágenes con detecciones incorrectas resaltadas para identificar patrones de error.

In [ ]:
# Analizar predicciones con baja confianza
if MODELS and Path(MODELS[0]).exists():
    model = YOLO(MODELS[0])
    
    # Detectar en validación con umbral bajo para ver falsos positivos
    val_results = model.val(data=DATASET_YAML, conf=0.1)
    
    print('\nMétricas con umbral bajo (conf=0.1):')
    print(f'  Precision: {val_results.box.mp:.4f}')
    print(f'  Recall:    {val_results.box.mr:.4f}')
    print('\nℹ️ Un recall alto con precision baja indica muchos falsos positivos')
    print('   Un recall bajo indica que el modelo no detecta algunos vehículos')

## Siguiente Paso

- Si las métricas son satisfactorias: **04_fine_tuning_semic.ipynb**
- Si necesitan mejorar: Ajustar hiperparámetros en `configs/training/improved.yaml` y re-entrenar